# Evaluation: Compare Laptop vs Colab

After each run, **6 files** are available in `results/` (separator **`;`**, UTF-8 with BOM):

| File | Content |
|------|---------|
| `metrics.csv` | **Primary comparison file**: Accuracy, Precision, Recall, F1 (overall, gender, emotion, age) |
| `evaluation_results.csv` | Per image: all columns |
| `predictions.csv` | Per image: compact text triplets |
| `confusion_gender.csv` | Gender confusion matrix |
| `confusion_age.csv` | Age-group confusion matrix |
| `confusion_emotion.csv` | Emotion confusion matrix |

**Tip:** For a quick summary table, **`metrics.csv` + runtime metrics** from both environments are usually enough.

Workflow:
1. Laptop: zip the `results/` folder -> extract as `results_laptop/`.
2. Colab: run the same evaluation -> download `results/` -> store as `results_colab/`.
3. Set `PATH_LAPTOP` and `PATH_COLAB` below (or upload both folders in Colab).

In [ ]:
import pandas as pd
from pathlib import Path

# Adjust: folders containing each results CSV set
PATH_LAPTOP = Path("results_laptop")   # e.g. after extraction
PATH_COLAB = Path("results_colab")

SEP = ";"
ENC = "utf-8-sig"


def load_metrics(folder: Path) -> pd.DataFrame:
    p = folder / "metrics.csv"
    return pd.read_csv(p, sep=SEP, encoding=ENC)


def load_all(folder: Path) -> dict[str, pd.DataFrame]:
    return {
        "metrics": pd.read_csv(folder / "metrics.csv", sep=SEP, encoding=ENC),
        "evaluation_results": pd.read_csv(folder / "evaluation_results.csv", sep=SEP, encoding=ENC),
        "predictions": pd.read_csv(folder / "predictions.csv", sep=SEP, encoding=ENC),
        "confusion_gender": pd.read_csv(folder / "confusion_gender.csv", sep=SEP, encoding=ENC),
        "confusion_age": pd.read_csv(folder / "confusion_age.csv", sep=SEP, encoding=ENC),
        "confusion_emotion": pd.read_csv(folder / "confusion_emotion.csv", sep=SEP, encoding=ENC),
    }

## 1) Core comparison: `metrics.csv` side by side

In [ ]:
m_lap = load_metrics(PATH_LAPTOP).assign(env="Laptop")
m_col = load_metrics(PATH_COLAB).assign(env="Colab")

# One table: rows = scope, columns = metric, two environments
key_cols = ["scope", "accuracy", "precision", "recall", "f1"]
lap = m_lap[key_cols].rename(columns={c: f"{c}_laptop" for c in key_cols if c != "scope"})
col = m_col[key_cols].rename(columns={c: f"{c}_colab" for c in key_cols if c != "scope"})
compare = lap.merge(col, on="scope", how="outer")
compare

## 2) Add runtime metrics (manual or from logs)

In Colab, for example use `%%time` on the evaluation cell, or use `total_runtime_s` from `metrics.csv` (same value in each row - total run time).

Below: display `mean_inference_time_s` from `metrics`.

In [ ]:
time_cols = ["scope", "mean_inference_time_s", "std_inference_time_s", "total_runtime_s", "n_images"]
display(m_lap[time_cols])
display(m_col[time_cols])

## 3) Optional: confusion matrices as numeric tables (without plots)

The first column is usually `true_*`; the remaining columns are predicted counts.

In [ ]:
lap_all = load_all(PATH_LAPTOP)
col_all = load_all(PATH_COLAB)

for name in ["confusion_gender", "confusion_age", "confusion_emotion"]:
    print("===", name, "LAPTOP ===")
    display(lap_all[name])
    print("===", name, "COLAB ===")
    display(col_all[name])